In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
tokenizer.pad_token = tokenizer.eos_token
tokenizer

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


GPT2TokenizerFast(name_or_path='microsoft/DialoGPT-medium', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}

In [3]:
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)

In [4]:
dataset = pd.read_csv("normalized_context_and_response.csv")
dataset.head()

,context,response
0,i'm going through some things with my feelings...,if everyone thinks you're worthless then maybe...
1,i'm going through some things with my feelings...,hello and thank you for your question and seek...
2,i'm going through some things with my feelings...,first thing i'd suggest is getting the sleep y...
3,i'm going through some things with my feelings...,therapy is essential for those that are feelin...
4,i'm going through some things with my feelings...,i first want to let you know that you are not ...


In [5]:
contexts = dataset['context'].astype('str').values
contexts[:5]

array(["i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthless to everyone",
       "i'm going through some things with my feelings and myself i barely sleep and i do nothing but think about how i'm worthless and how i shouldn't be here\r\n   i've never tried or contemplated suicide i've always wanted to fix my issues but i never get around to it\r\n   how can i change my feeling of being worthle

In [6]:
responses = dataset['response'].astype('str').values
responses[:5]

array(["if everyone thinks you're worthless then maybe you need to find new people to hang out withseriously the social context in which a person lives is a big influence in self-esteemotherwise you can go round and round trying to understand why you're not worthless then go back to the same crowd and be knocked down againthere are many inspirational messages you can find in social media \xa0maybe read some of the ones which state that no person is worthless and that everyone has a good purpose to their lifealso since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terriblebad feelings are part of living \xa0they are the motivation to remove ourselves from situations and relationships which do us more harm than goodbad feelings do feel terrible \xa0 your feeling of worthlessness may be good in the sense of motivating you to find out that you are much better than your feelings today",
       "hello and thank you for you

In [7]:
def combineText(example):
    return {
        "text": "User: " + example['context'] + 
            "Bot: " + example['response']
    }

In [8]:
def encode(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

In [9]:
def add_labels(example):
    example['labels']=example['input_ids']
    return example

In [10]:
datasets = Dataset.from_dict({
    "context": contexts,
    "response": responses
})
datasets

Dataset({
    features: ['context', 'response'],
    num_rows: 3512
})

In [11]:
datasets_split = datasets.train_test_split(test_size=0.2)
datasets_split

DatasetDict({
    train: Dataset({
        features: ['context', 'response'],
        num_rows: 2809
    })
    test: Dataset({
        features: ['context', 'response'],
        num_rows: 703
    })
})

In [12]:
trainSet = datasets_split['train']
trainSet

Dataset({
    features: ['context', 'response'],
    num_rows: 2809
})

In [13]:
testSet = datasets_split['test']
testSet

Dataset({
    features: ['context', 'response'],
    num_rows: 703
})

In [14]:
trainSet = trainSet.map(combineText)
trainSet

Map: 100%|██████████| 2809/2809 [00:00<00:00, 6718.68 examples/s]


Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 2809
})

In [15]:
testSet = testSet.map(combineText)
testSet

Map: 100%|██████████| 703/703 [00:00<00:00, 6924.95 examples/s]


Dataset({
    features: ['context', 'response', 'text'],
    num_rows: 703
})

In [16]:
trainSet = trainSet.map(encode, batched=True)
trainSet

Map: 100%|██████████| 2809/2809 [00:00<00:00, 4074.93 examples/s]


Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 2809
})

In [17]:
testSet = testSet.map(encode, batched=True)
testSet

Map: 100%|██████████| 703/703 [00:00<00:00, 3016.43 examples/s]


Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask'],
    num_rows: 703
})

In [18]:
trainSet = trainSet.map(add_labels)
trainSet

Map: 100%|██████████| 2809/2809 [00:00<00:00, 4249.58 examples/s]


Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2809
})

In [19]:
testSet = testSet.map(add_labels)
testSet

Map: 100%|██████████| 703/703 [00:00<00:00, 3640.81 examples/s]


Dataset({
    features: ['context', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 703
})

In [20]:
trainingArgs = TrainingArguments(
    output_dir="./output",
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=None
)

In [21]:
trainer=Trainer(
    model=model,
    args=trainingArgs,
    train_dataset=trainSet,
    eval_dataset=testSet
)

In [ ]:
trainer.evaluate(testSet)

 64%|██████▎   | 7/11 [08:18<05:05, 76.39s/it]

In [24]:
trainer.predict(testSet.select(range(80)))

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 45%|████▌     | 5/11 [07:57<09:32, 95.47s/it] 


PredictionOutput(predictions=array([[[ -8.450652  , -13.837666  , -15.53783   , ..., -13.0114155 ,
         -13.09387   ,  -5.050879  ],
        [ -6.3937106 , -13.606663  , -12.112084  , ..., -10.328847  ,
          -9.338417  ,   2.3756447 ],
        [ -3.6948907 , -11.887741  , -11.707344  , ...,  -7.3320513 ,
          -7.05541   ,   4.6809363 ],
        ...,
        [  2.4968567 ,  -8.866075  ,  -8.555484  , ...,  -1.990624  ,
          -0.51444864,  15.428238  ],
        [  7.034876  ,  -4.9308043 ,  -4.047919  , ...,   0.6049278 ,
           2.8740568 ,  19.261162  ],
        [  0.80375004,  -6.708885  ,  -6.5483937 , ...,   0.6851144 ,
           0.7940072 ,  19.626705  ]],

       [[ -8.450652  , -13.837666  , -15.53783   , ..., -13.0114155 ,
         -13.09387   ,  -5.050879  ],
        [ -6.3937106 , -13.606663  , -12.112084  , ..., -10.328847  ,
          -9.338417  ,   2.3756447 ],
        [ -1.5513605 ,  -6.656332  ,  -8.2259865 , ...,  -7.048557  ,
          -6.708316  ,

In [ ]:
trainer.train()